# Executive Summary

## Project Overview
This project develops a Machine Learning solution to predict individual medical insurance charges using customer demographic and lifestyle information. Multiple regression algorithms are trained and evaluated to identify the model that provides the most accurate predictions.

## Objective
- Predict medical insurance charges using historical customer data.
- Train and evaluate multiple regression models.
- Compare model performance using regression metrics.
- Select the best-performing model for deployment.

## Approach
1. Data Loading
2. Exploratory Data Analysis (EDA)
3. Data Preprocessing
4. Feature Engineering and Encoding
5. Train-Test Split
6. Model Training
7. Model Evaluation and Comparison
8. Best Model Selection
9. Model Serialization for Deployment

## Outcome
The project compares multiple regression models and selects the best-performing model based on evaluation metrics, demonstrating an end-to-end machine learning workflow suitable for real-world business applications.

# Dataset Description

| Feature | Type | Description |
|---|---|---|
| age | Numerical | Age of the policyholder |
| sex | Categorical | Gender |
| bmi | Numerical | Body Mass Index |
| children | Numerical | Number of dependents |
| smoker | Categorical | Smoking status |
| region | Categorical | Residential region |
| charges | Target | Annual medical insurance charges |


# 📚 Import Libraries

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# 📂 Load Dataset

In [2]:
# Load the input data to pandas dataframe.
df = pd.read_csv("../data/insurance_pre.csv")
df.head()

,age,sex,bmi,children,smoker,charges
0,19,female,27.900,0,yes,16884.92400
1,18,male,33.770,1,no,1725.55230
2,28,male,33.000,3,no,4449.46200
3,33,male,22.705,0,no,21984.47061
4,32,male,28.880,0,no,3866.85520


# 🔍 Exploratory Data Analysis (EDA)

In [3]:
# Get the number of rows and columns in the dataset.
df.shape

(1338, 6)

- There are 1338 rows and 6 coluns in the dataset.

In [4]:
# Get the information about the datatype of each column in the dataset.
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1338 entries, 0 to 1337
Data columns (total 6 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       1338 non-null   int64  
 1   sex       1338 non-null   object 
 2   bmi       1338 non-null   float64
 3   children  1338 non-null   int64  
 4   smoker    1338 non-null   object 
 5   charges   1338 non-null   float64
dtypes: float64(2), int64(2), object(2)
memory usage: 62.8+ KB


- There are fours numberical columns (two columns with interger data and two columns with floating data)
- Two columns with categorical data ("sex" and "smoker" are nominal type of data).

In [5]:
# Check the missing values in the columns of the dataset.
df.isnull().sum()

age         0
sex         0
bmi         0
children    0
smoker      0
charges     0
dtype: int64

- There is no empty value in the entire dataset.

In [6]:
# Check if there is any duplicate in the dataset.
df.duplicated().sum()

np.int64(1)

- There is one row with duplicate value in the dataset.

In [7]:
# Display the duplicate rows in the dataset.
df[df.duplicated(keep=False)]

,age,sex,bmi,children,smoker,charges
195,19,male,30.59,0,no,1639.5631
581,19,male,30.59,0,no,1639.5631


In [8]:
# Remove the duplicate row(s) in the dataset.
df = df.drop_duplicates()

# 🧹 Data Preprocessing

In [9]:
# Convert the nominal data in the columns such as "sex" and "smoker".
df = pd.get_dummies(df, columns=['sex', 'smoker'], dtype=int, drop_first=True)

In [10]:
# Print first five rows to show the nominal values are converted to integers.
df.head()

,age,bmi,children,charges,sex_male,smoker_yes
0,19,27.900,0,16884.92400,0,1
1,18,33.770,1,1725.55230,1,0
2,28,33.000,3,4449.46200,1,0
3,33,22.705,0,21984.47061,1,0
4,32,28.880,0,3866.85520,1,0


# # 🎯 Define Features and Target

In [11]:
X = df.drop("charges", axis=1)
y = df["charges"]

# ✂️ Train-Test Split

In [12]:
# Import train_test_split function from sklearn library.
from sklearn.model_selection import train_test_split

# Split the dataset into 70% for training and 30% for testing.
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.3,random_state=42)

# Regression Models

## 1. Multiple Linear Regression model

In [13]:
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
model = LinearRegression()

# Standardization for scale-sensitive models
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Train the model
model.fit(X_train_scaled, y_train)

# Apply the trained model on unseen input data.
y_pred = model.predict(X_test_scaled)

# Calculate R2 score.
from sklearn.metrics import r2_score

r2 = r2_score(y_test, y_pred)

# display R2 score.
print(f"r2 of multiple linear regression: {r2}")

r2 of multiple linear regression: 0.7708695009612091


## 2. Support Vector Regression model

In [14]:
import numpy as np
from sklearn.svm import SVR
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import StandardScaler

# Create an object of the SVR class
model = SVR()

# Define the hyperparameter values to search
param_grid = {
    "kernel": ["linear", "poly", "rbf", "sigmoid"],
    "C": [0.01, 0.1, 1.0, 10, 100]
}

# Create the GridSearchCV object
grid = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    cv=5,
    scoring="r2",
    n_jobs=-1
)
# Standardization for scale-sensitive models
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Train the model using all hyperparameter combinations
grid.fit(X_train_scaled, y_train.to_numpy().ravel())

# Display the best hyperparameters
print("Best Parameters:")
print(grid.best_params_)

# Display the best cross-validation score
print("\nBest Cross-Validation R² Score:")
print(grid.best_score_)

# Retrieve the best trained model
best_model = grid.best_estimator_

Best Parameters:
{'C': 100, 'kernel': 'linear'}

Best Cross-Validation R² Score:
0.593599819036111


## 3. Decision Tree model

In [15]:
import numpy as np
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import GridSearchCV

# Create an instance of DecisionTreeRegressor
model = DecisionTreeRegressor(random_state=42)

# Define the hyperparameter values to search
param_grid = {
    "criterion": ["squared_error", "absolute_error", "poisson"],
    "splitter": ["best", "random"],
    "max_features": [2, 4, 0.5, 0.8, "sqrt", "log2", None]
}

# Create the GridSearchCV object
grid = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    cv=5,
    scoring="r2",
    n_jobs=-1
)

# Train the model using all hyperparameter combinations
grid.fit(X_train, y_train)

# Display the best hyperparameters
print("Best Parameters:")
print(grid.best_params_)

# Display the best cross-validation score
print("\nBest Cross-Validation R² Score:")
print(grid.best_score_)

# Retrieve the best trained model
best_model = grid.best_estimator_

Best Parameters:
{'criterion': 'poisson', 'max_features': 4, 'splitter': 'best'}

Best Cross-Validation R² Score:
0.6875254352977329


## 4. Random Forest model

In [16]:
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV

# Create an instance of RandomForestRegressor
model = RandomForestRegressor(random_state=0)

# Define the hyperparameter values to search
param_grid = {
    "criterion": ["squared_error", "absolute_error", "poisson"],
    "n_estimators": range(10, 50, 2)
}

# Create the GridSearchCV object
grid = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    cv=5,
    scoring="r2",
    n_jobs=-1
)

# Train the model using all hyperparameter combinations
grid.fit(X_train, np.ravel(y_train))

# Display the best hyperparameters
print("Best Parameters:")
print(grid.best_params_)

# Display the best cross-validation score
print("\nBest Cross-Validation R² Score:")
print(grid.best_score_)

# Retrieve the best trained model
best_model = grid.best_estimator_

Best Parameters:
{'criterion': 'poisson', 'n_estimators': 46}

Best Cross-Validation R² Score:
0.8198315416529003


# Save The Best Model

In [17]:
## Save the model to pickle library.
import pickle

# Save the best trained model to a pickle file
with open("../models/final_model.sav", "wb") as file:
    pickle.dump(best_model, file)

print("Model saved successfully!")

Model saved successfully!


# Model Comparison

| Model | R² Score | Selected |
|---|---:|:---:|
| Multiple Linear Regression | 0.7709 | |
| Support Vector Regression (SVR) | 0.5936 | |
| Decision Tree Regressor | 0.6875 | |
| Random Forest Regressor | **0.8198** | ✅ |



# Final Conclusion with Business Insights

## Conclusion
Among the evaluated regression models, **Random Forest Regressor** achieved the highest R² score (0.8198), making it the best-performing model for predicting medical insurance charges in this project.

## Business Insights
- Supports more accurate premium estimation.
- Helps identify high-cost customers.
- Enables data-driven pricing decisions.
- Reduces manual effort in insurance cost estimation.
